In [30]:
from typing import NamedTuple
from dataclasses import dataclass

@dataclass(frozen=True)
class Vector2D:
    x: float
    y: float

    def __post_init__(self):
        object.__setattr__(self, 'x', float(self.x))
        object.__setattr__(self, 'y', float(self.y))
        

    def __add__(self, other: Vector2D):
        return Vector2D(self.x + other.x, self.y + other.y)

    def __bool__(self):
        return bool(abs(self))

    def __str__(self):
        return f'({self.x}, {self.y})'

    def __bytes__(self):
        return bytes([self.x, self.y])

    def __abs__(self) -> float:
        return (self.x**2 + self.y**2)**.5

    def __iter__(self):
        return (i for i in (self.x, self.y))

    

v1 = Vector2D(3, 4)
bool(v1)

True

In [91]:
import math
from array import array


class Vector2D:
    typecode = 'd'
    __match_args__ = ('x', 'y') # this enables positional pattern matching!

    def __init__(self, x: int | float, y: int | float):
        # __xxx to create a private attribute
        self.__x = float(x) # converting to float catches errors early
        self.__y = float(y)

    @property # readonly property!
    def x(self):
        return self.__x

    @property
    def y(self):
        return self.__y

    def __iter__(self):
        #return (i for i in (self.x, self.y))
        yield self.x
        yield self.y

    def __repr__(self):
        classname = type(self).__name__
        return f'{classname}({self.x}, {self.y})'

    def __str__(self):
        return str(tuple(self))

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

    def __bytes__(self):
        return bytes([ord(self.typecode)]) + bytes(array(self.typecode, self))

    def __eq__(self, other: Vector2D):
        return tuple(self) == tuple(other)

    @classmethod
    def frombytes(cls, octets) -> Vector2D:
        '''An alternative constructor: fromxxx()'''
        typecode = chr(octets[0])
        memv = memoryview(octets[1:])(typecode)
        return cls(*memv)

    def __hash__(self):
        return hash((self.x, self.y))

    def __int__(self):
        return int(max(self.x, self.y))
    

In [92]:
v1 = Vector2D(3, 4)
int(v1)

4

In [46]:
x, y = v1
x, y

(3.0, 4.0)

In [47]:
bytes(v1)

b'd\x00\x00\x00\x00\x00\x00\x08@\x00\x00\x00\x00\x00\x00\x10@'

In [59]:
v2 = Vector2D(1, 5)
print(hash(v1), hash(v2))

{v1, v2}

1079245023883434373 173794974761290439


{Vector2D(1.0, 5.0), Vector2D(3.0, 4.0)}

### @classmethod vs @staticmethod
1. `@classmethod` always receives the class as its first argument. It's common to decorate an alternative constructor.
    - Support of chaining @classmethod with other decorators is removed in 3.13.
2. `@staticmethod` doesn't need to receive class as its first argument. Its use shall be rare.

## Private and Protected attributes
1. Prefixing an attribute with double underscore causes the interpreter to add the attribute to the class `__dict__` prefixing with a single underscore followed by a class name. This is a safety not security way of indicating private as there is no real way to fully make an attribute private as one can still overwrite this if they want to. 
2. Some people don't like this and by convention it's using a single underscore to indicate privateness, and sometimes this is called protecting the attribute.

## Saving Memory with `__slots__`
1. By default, instance attributes are stored in `__dict__` attribute but dict has heavy memory footprint. 
2. To save memory, put instance attributes into a `__slot__` class attribute which can either be a list or a tuple though tuple is preferred; this causes those instance attributes to be stored in a hidden array or reference that uses less memory than `__dict__`.
3. slots must be present when the class is created; adding or changing it later has no effect.
    - Once slots is defined, unless `__dict__` is put into slots, no dynamic attribute can be added to the instance. Though doing so can defeat the purpose of having slots in the first place.  
    - Subclass by default will inherite the base class slot; to have its own slot, it has to be defined explicitly. 
    - Classes having slots can't use `@cached_property` decorator unless `__dict__` is put into slots. 
    - Instance can't be the targets of weak references unless `__weakref__` to the slots. 
4. Slots not only makes an instance use less memory but creating such instances is also faster.

In [68]:
class Pixel:
    __slots__ = ('x', 'y') #

p = Pixel()
p.__dict__ # no such attribute as we defined __slots__.

AttributeError: 'Pixel' object has no attribute '__dict__'

In [ ]:
p.x # can't be read before being set

AttributeError: 'Pixel' object has no attribute 'x'

In [70]:
# setting slots attributes are fine
p.x = 10
p.y = 20
p.x, p.y

(10, 20)

In [ ]:
p.color = 'red' # no dynamic attribute is allowed

AttributeError: 'Pixel' object has no attribute 'color' and no __dict__ for setting new attributes

In [74]:
class OpenPixel(Pixel):
    pass

op = OpenPixel()
print(op.__dict__) # it does have this attribute
op.color = 'green' # so setting a dynamic attribute works
print(op.__dict__) 

{}
{'color': 'green'}


In [ ]:
op.__slots__ # it inheriates base class slots.

('x', 'y')

In [77]:
op.x = 10 # setting a slot attribute will not appear in dict!
op.__dict__, op.x

({'color': 'green'}, 10)

In [80]:
class ColorPixel(Pixel):
    __slots__ = ('color', ) # a subclass has its own slots.

cp = ColorPixel()
cp.__slots__

('color',)

In [81]:
cp.__dict__ # no dict anymore because it has its own slots.

AttributeError: 'ColorPixel' object has no attribute '__dict__'

In [ ]:
from copy import copy

cp.color = 'green'
cp_copy = copy(cp) # works with copy out of box since 3.11
cp_copy.color


'green'

## Overriding Class Attribute
1. By default, a class attribute value can be used as the default value for instance attribute.
2. But this can be overridden by creating an instance attribute of the same name and assign a value to it.
3. While the class attribute is untouched, retrieving such an instance attribute will be reading from the value on the instance. 
4. And the class attribute value can be changed too via `ClsName.attribute = new_value` and this will affect all instances reading this attribute while itself doesn't define its own.
5. But the common practice is to subclass just to customize a class data attribute to make this change explicit!
    - This has the benefits of setting a new default value for all instances of the subclass rather than an one-off change on a base class instance directly.

In [86]:
class Base:
    x = 10 # a class attribute

class Sub1(Base):
    pass

class Sub2(Base):
    x = 100

b = Base()
s1 = Sub1()
s2 = Sub2()
print(b.x, s1.x, s2.x)
b.x = -1
print(b.x, s1.x, s2.x)
s1.x = 20
print(b.x, s1.x, s2.x)
Base.x = -10
s3 = Sub1()
print(b.x, s1.x, s2.x, s3.x)



10 10 100
-1 10 100
-1 20 100
-1 20 100 -10
